In [0]:
%pip install pandas openpyxl

In [0]:
import pandas as pd
import shutil
from datetime import datetime

In [0]:
spark_df = spark.table("prd_mega.sgpbpi163.3a_goat_master")
goat_data = spark_df.toPandas()
display(spark_df)

In [0]:
pandas_df = pd.read_csv("/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/Procurement Step/Procurement_Step_Data_08_18_2026.csv")
display(pandas_df)

In [0]:
goat_data = spark_df.toPandas()

In [0]:
subset_df = pandas_df[['proj_id', 'cntrct_desc', 'cntrct_sign_date', 'cntrct_amt', 'supplr_cntry_name']].dropna(subset=['cntrct_desc'])
display(subset_df)

In [0]:
subset_df['data model'] = (
    subset_df['cntrct_desc'] +
    " (Date: " + subset_df['cntrct_sign_date'].astype(str) +
    ", Supplier Name: " + subset_df['supplr_cntry_name'].str.replace(r"\s*\(.*\)", "", regex=True).fillna('') +
    ", Supplier Country: " + subset_df['supplr_cntry_name'].str.extract(r"\((.*?)\)", expand=False).fillna('') +
    ", Contract Amount:  $" + subset_df['cntrct_amt'].astype(str) +
    ")"
)
display(subset_df)

In [0]:
procurement_steps_cleaned = (
    subset_df
    .sort_values(['proj_id', 'cntrct_sign_date'])
    .groupby('proj_id')
    .agg({'data model': lambda x: ' '.join([f"{i+1}. {v}" for i, v in enumerate(x)])})
    .reset_index()
)
display(procurement_steps_cleaned)

In [0]:
unique_values = {"proj_id": procurement_steps_cleaned["proj_id"].unique()}
df_shape = procurement_steps_cleaned.shape

print(len(procurement_steps_cleaned["proj_id"].unique()))
print("Shape of procurement_steps_cleaned:", df_shape)

In [0]:
current_date = datetime.now().strftime("%d%m")
output_path = f"/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/Procurement Step/procurement_data_cleaned_{current_date}"

spark_cleaned_df = spark.createDataFrame(procurement_steps_cleaned)
local_path = f"/tmp/goat_with_procurement_{current_date}.xlsx"
volume_path = f"{output_path}.xlsx"

procurement_steps_cleaned.to_excel(local_path, index=False)
shutil.copy(local_path, volume_path)
dbutils.fs.cp(f"file:{local_path}", volume_path)

In [0]:
procurement_steps_cleaned = procurement_steps_cleaned.rename(columns={"proj_id": "PROJ_ID", "data model": "ProcurementSteps"}) 
df_result_cleaned = goat_data.merge(
    procurement_steps_cleaned,
    on="PROJ_ID",
    how="left"
)
display(df_result_cleaned)

In [0]:
current_date = datetime.now().strftime("%d%m")
output_path = f"/Volumes/prd_mega/sgpbpi163/vgpbpi163/GOAT/Procurement Step/goat_with_procurement_{current_date}"

In [0]:
local_path = f"/tmp/goat_with_procurement_{current_date}.xlsx"
volume_path = f"{output_path}.xlsx"

df_result_cleaned.to_excel(local_path, index=False)
shutil.copy(local_path, volume_path)
df_result_cleaned.to_excel(local_path, index=False)
dbutils.fs.cp(f"file:{local_path}", volume_path)